# W02B · 언어 모델의 활용 사례

교재 2.1의 토큰화 → 다음 토큰 확률 → 생성 → 제로샷·퓨샷을 실험하고,
마지막에 같은 기능을 웹 앱으로 연결한다. [강의노트](https://github.com/lunalab-ai/genAI/blob/main/course/notion/w02b/w02b-language-model-applications.md)

Colab에서 내 Drive에 사본을 저장하고 위에서부터 실행한다. 공개 모델 약 1 GB를
처음 다운로드하며 유료 API 키·개인 파일 업로드는 필요 없다. 모델은 한 번 로드한다.
짧은 영어 예문을 기본으로 하고 한국어 토큰화도 비교한다. 이번 자료에는 성적 반영 과제가 없다.

**0.1.1 수정본:** 기존 오류가 난 런타임은 연결 해제·삭제 후 새로 연결한다.
첫 셀은 호환 가능한 PyTorch를 유지하며, 텍스트 실습에 필요 없는 영상·음성 패키지는
import가 실패할 때에만 제거한다. 설치가 끝난 뒤 두 번째 코드 셀에서 실제 모델을 준비한다.


In [ ]:
import sys
import subprocess
from pathlib import Path
from importlib.metadata import version, PackageNotFoundError

PUBLIC_REF = "genai-lab-v0.1.1"
PACKAGE_URL = f"git+https://github.com/lunalab-ai/genAI.git@{PUBLIC_REF}#subdirectory=src"
try:
    ready = version("luna-genai") == "0.1.1"
    ready = ready and version("transformers") == "5.15.1" and version("gradio") == "6.26.0"
    ready = ready and (2, 8) <= tuple(map(int, version("torch").split(".")[:2])) < (3, 0)
except (ValueError, PackageNotFoundError):
    ready = False
if not ready:
    if any(name in sys.modules for name in ("luna_genai", "transformers", "gradio")):
        raise RuntimeError("이전 라이브러리가 메모리에 남아 있습니다. 런타임을 연결 해제·삭제하고 첫 셀부터 실행하세요.")
    # 저장소 자체를 내려받아 실행 중이면 그 체크아웃을 설치한다.
    # Colab의 GitHub notebook 열기는 파일 하나만 열므로 고정 공개 tag로 설치한다.
    local_package = Path("src/pyproject.toml")
    source = str(local_package.parent.resolve()) if local_package.exists() else PACKAGE_URL
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", source])
from luna_genai.runtime import prepare_text_runtime
removed = prepare_text_runtime(repair_optional=True)
if removed and any(name in sys.modules for name in ("transformers", "torchvision", "torchaudio", "torchcodec")):
    raise RuntimeError("패키지 충돌을 복구했습니다. 런타임을 다시 시작하고 첫 셀부터 실행하세요.")
print("공통 패키지 준비 완료 ·", PUBLIC_REF)


## 1. 데이터와 모델 준비

`examples` 2개는 퓨샷 프롬프트용, `development` 4개는 조정용,
`evaluation` 4개는 마지막 비교용이다. 모두 수업용으로 직접 작성한 영화 리뷰이며
교재 원자료나 IMDb에서 가져온 데이터가 아니다. 패키지가 데이터 파일을 함께 제공한다.

`LanguageModelLab`은 학습이 끝난 기본 모델을 로드한다. `.fit()`이나 역전파는 수행하지 않는다.
준비가 실패하면 원인을 출력하고 수치 실험만 진행한다. 해당 출력은 실제 모델 예측이 아니다.


In [ ]:
import pandas as pd
import numpy as np
import torch
from IPython.display import display
from luna_genai import (LanguageModelLab, load_reviews, filtered_distribution,
                        distribution_table, build_app, MODEL_ID, MODEL_REVISION)

development = load_reviews("development")
display(pd.DataFrame(development))
lab = None
try:
    lab = LanguageModelLab()
    print("실제 모델:", MODEL_ID, "revision:", MODEL_REVISION, "device:", lab.device)
except (ImportError, OSError, RuntimeError) as exc:
    print("모델 준비 실패:", str(exc))
    print("현재는 수치 실험 전용 모드입니다. 연결 복구 후 이 셀을 다시 실행하세요.")


## 2. 토큰을 예상하고 관찰하기

먼저 `cat`과 앞 공백이 있는 ` cat`의 ID가 같을지 예상한다.
토큰 수는 단어 수와 항상 같지 않다. 내부 토큰 표기에는 공백·바이트를 나타내는 기호가 있다.
한국어 토큰의 개별 decode에 `�`가 나타나면 전체 ID를 한 번에 decode한 결과와 비교한다.


In [ ]:
text_main = "A cat naps."
if lab is not None:
    for text in [text_main, "cat", " cat", "고양이가 잠을 잡니다."]:
        print("입력:", repr(text))
        display(lab.token_table(text))
    ids_main = lab.tokenizer.encode(text_main, add_special_tokens=False)
    print("전체 ID 복원:", lab.tokenizer.decode(ids_main))


### 빈칸 문제 A · 원문 복원

**목표:** ID 목록을 문자열로 복원한다. `recovered_student`의 None을 바꾼다.
**힌트:** 토크나이저의 decode는 ID 목록을 받는다.
**자가 점검:** 결과가 `text_student`와 같은지 확인한다.
문제용 변수는 뒤 실험과 독립적이다. None인 채로도 전체 실행은 계속된다.


In [ ]:
text_student = "A bird sings."
if lab is not None:
    ids_student = lab.tokenizer.encode(text_student, add_special_tokens=False)
    recovered_student = None  # TODO: 전체 ID를 문자열로 복원
    if recovered_student is None:
        print("문제 A: 빈칸을 채운 뒤 다시 실행하세요.")
    else:
        assert recovered_student == text_student
        print("문제 A: 원문 복원 확인")


## 3. 다음 후보의 확률 관찰

로짓은 정규화 전 점수이다. `last_logits`는 문장 하나의 마지막 위치에서
전체 어휘 후보에 대한 점수를 가져온다. softmax의 분모는 전체 어휘이며,
상위 8개만 보이는 표의 확률 합은 1보다 작을 수 있다.
아래 배정밀도 계산은 작은 반올림 오차를 줄이기 위한 표시용 계산이다.


In [ ]:
if lab is not None:
    logits_main = lab.last_logits("The movie was")
    probabilities_main = torch.softmax(logits_main.double(), dim=-1)
    print("어휘 후보 수:", len(logits_main), "전체 확률 합:", float(probabilities_main.sum()))
    for prompt in ["The movie was", "The movie was boring and"]:
        candidates_main = lab.next_tokens(prompt)
        print("문맥:", prompt, "상위 8개 확률 합:", candidates_main["전체 어휘 확률"].sum())
        display(candidates_main)


### 디버깅 문제 B · 축과 마지막 위치

**목표:** `[배치, 위치, 어휘]`에서 다음 토큰에 해당하는 위치 선택.
아래는 실제 모델 출력이 아닌 모양 확인용 작은 배열이다. `wrong_student`는 첫 위치를 읽는다.
**힌트:** Python에서 `-1`은 마지막 위치이다.
**자가 점검:** 결과는 후보 4개의 벡터이고 첫 위치의 벡터와 달라야 한다.


In [ ]:
fake_logits = np.arange(12).reshape(1, 3, 4)
wrong_student = fake_logits[0, 0, :]
last_student = None  # TODO: 마지막 위치의 후보 벡터
if last_student is None:
    print("문제 B: 축의 의미를 설명한 뒤 수정하세요.")
else:
    assert np.asarray(last_student).shape == (4,)
    assert np.array_equal(last_student, fake_logits[:, -1, :].reshape(-1))
    assert not np.array_equal(last_student, wrong_student)
    print("문제 B: 마지막 위치 확인")


## 4. 확률 수치 실험 → 실제 생성

먼저 가상 로짓 `[3,2,1,0]`을 사용한다. 실제 Qwen 결과가 아니다.
T만 비교할 때는 top-k=0, top-p=1로 필터를 해제한다.
top-p는 누적 확률이 임계값을 처음 넘기는 후보까지 남긴다.


In [ ]:
for t in [0.5, 1.0, 2.0]:
    print("가상 수치 실험 · temperature:", t)
    display(distribution_table(t))
display(distribution_table(temperature=1, top_k=0, top_p=0.8))
assert np.isclose(filtered_distribution([3, 2, 1, 0], top_p=.8).sum(), 1)


실제 생성은 선택한 토큰을 문맥에 붙여 반복한다. 아래 반환값은 시작 문장을 제외한 생성 부분이다.
먼저 문장이 어떻게 이어질지 예상한다. 생성 문장이 엉뚱하거나 반복되면 그대로 기록한다.
`greedy`는 가장 높은 후보를 선택하며 sample용 설정을 사용하지 않는다.


In [ ]:
prompt_main = "A small robot opened the door and"
generation_records = []
if lab is not None:
    settings = [
        ("greedy", dict(strategy="greedy")),
        ("T=0.5", dict(strategy="sample", temperature=.5, top_k=0, top_p=1)),
        ("T=1.2", dict(strategy="sample", temperature=1.2, top_k=0, top_p=1)),
        ("T=0.8, p=1", dict(strategy="sample", temperature=.8, top_k=0, top_p=1)),
        ("T=0.8, p=0.8", dict(strategy="sample", temperature=.8, top_k=0, top_p=.8)),
    ]
    for name, setting in settings:
        output = lab.generate(prompt_main, seed=7, max_new_tokens=20, **setting)
        generation_records.append({"조건": name, "생성 부분": output})
    display(pd.DataFrame(generation_records))
    repeated = lab.generate(prompt_main, strategy="sample", temperature=.8,
                            top_k=0, top_p=.8, seed=7, max_new_tokens=20)
    print("같은 설정·seed 재실행 일치:", repeated == generation_records[-1]["생성 부분"])


### 비교 문제 C

**목표:** 통제한 조건과 변경한 조건을 구분한다.
위 표에서 temperature만 비교할 수 있는 두 행을 선택하고, 다음에는 top-p만 비교한다.
**힌트:** seed·프롬프트·생성 길이도 같은지 확인한다.
**자가 점검:** “T가 높으면 항상 더 정확하다”처럼 관찰 범위를 넘는 결론을 쓰지 않는다.
여유가 있으면 `strategy="beam"`으로 짧은 생성을 추가한다. 폭 3, 길이 보정 0의 시연이다.


In [ ]:
comparison_note = {
    "비교한 두 조건": "",
    "바꾼 한 가지": "",
    "실제 관찰": "",
    "아직 확인하지 못한 것": "",
}
comparison_note


## 5. 제로샷과 2-shot 감성 분류

`few_shot=True`는 고정 예시 2개를 프롬프트에 더한다. 가중치 학습을 실행하지 않는다.
` positive`와 ` negative`는 앞 공백을 포함하여 단일 토큰인지 검사한다.
두 후보 내 비율과 전체 어휘 확률을 함께 읽는다. 상대 비율이 높아도 정답 확신도는 아니다.


In [ ]:
if lab is not None:
    print("실제 토크나이저에서 구한 레이블 ID:", lab.label_ids)
    review_main = development[0]["review"]
    for use_examples in [False, True]:
        result = lab.classify(review_main, few_shot=use_examples)
        print("2-shot:", use_examples, "예측:", result["prediction"], "두 레이블 전체 확률:", result["label_mass"])
        print(result["prompt"])
        display(result["scores"])


개발 문장으로 살펴본 뒤 마지막 비교에 평가 세트를 사용한다.
평가 문장을 보며 프롬프트를 계속 바꾸었다면 미사용 평가라고 부르지 않는다.
4개 모두 맞거나 두 방식이 같아도 일반 성능과 개선을 보장하지 않는다.


In [ ]:
evaluation_records = []
if lab is not None:
    for row in load_reviews("evaluation"):
        zero = lab.classify(row["review"])
        few = lab.classify(row["review"], few_shot=True)
        evaluation_records.append({**row, "zero_shot": zero["prediction"], "two_shot": few["prediction"],
                                   "zero_mass": zero["label_mass"], "two_mass": few["label_mass"]})
    evaluation_table = pd.DataFrame(evaluation_records)
    display(evaluation_table)
    for column in ["zero_shot", "two_shot"]:
        print(column, "맞힌 수 / 전체:", int((evaluation_table[column] == evaluation_table["label"]).sum()), "/", len(evaluation_table))


### 해석 문제 D · 오류를 찾거나 한계를 기록하기

**목표:** 단순 정답 수를 넘어 실패 원인을 제안한다.
부정어·반어·장단점을 섞은 리뷰 한 문장을 직접 쓰고 두 방식의 출력을 비교한다.
**힌트:** 예측과 함께 실제 프롬프트와 확률 질량도 읽는다.
**자가 점검:** 오분류를 찾지 못했으면 그렇게 기록한다. 결과를 꾸미지 않는다.


In [ ]:
review_student = ""  # TODO: 직접 작성한 짧은 영어 리뷰
interpretation_student = ""
if lab is not None and review_student.strip():
    for use_examples in [False, True]:
        result_student = lab.classify(review_student, few_shot=use_examples)
        print(use_examples, result_student["prediction"])
        display(result_student["scores"])


## 6. 마지막 통합 실습: 나의 웹 앱

공통 앱 빌더가 토큰·후보, 생성, 분류, 수치 실험 탭을 만든다. 이미 만든 lab을 전달한다.
여기서 모델을 다시 로드하지 않는다. 아래 확장 문제를 완성하면 나만의 입력/출력 탭을 추가한다.


In [ ]:
import gradio as gr
app = build_app(lab)
print("기본 앱 구성 완료 ·", "실제 모델 모드" if lab is not None else "수치 실험 전용 모드")


### 구현 문제 E · 토큰 수 callback과 화면 연결

**목표:** 입력 컴포넌트 → 공통 함수 → 반환값 → 출력 컴포넌트를 연결한다.
1. `my_callback`에 입력 문자열을 받아 토큰 수를 반환하는 함수를 작성한다.
2. 아래 연결 셀의 새 탭 안에 `gr.Textbox`, `gr.Button`, `gr.Number`를 만든다.
3. `button.click(my_callback, inputs=..., outputs=...)`로 연결한다.

**힌트:** `lab.token_table(text)`의 행 수를 사용하고 모델 로드는 함수 밖에 둔다.
**자가 점검:** 문장 두 개를 넣고, 표의 행 수와 앱의 숫자가 일치하는지 확인한다.
미완성이면 기본 앱만 실행한다. 완성 후 앱을 다시 띄울 때는 먼저 `app.close()`하고
기본 앱 구성 셀부터 다시 실행하여 탭을 중복 추가하지 않는다.


In [ ]:
my_callback = None  # TODO: def로 함수 작성 후 이 변수에 연결


In [ ]:
if my_callback is not None and lab is not None:
    with app:
        with gr.Tab("나의 토큰 수 패널"):
            gr.Markdown("입력과 출력 컴포넌트를 만들고 callback을 연결하세요.")
            # TODO: Textbox, Button, Number를 생성하고 click 연결


Colab에서는 아래 셀이 앱 화면과 공유 링크를 연다. 기본 앱의 두 탭 이상을 조작하고
입력·설정·출력 관계를 설명한다. 공유 링크는 실행 중인 런타임에 의존한다.
로컬 자동 검사에서는 서버를 계속 띄우지 않고 구성을 확인한 뒤 닫는다.
PC에서 실제 실행하려면 터미널에서 `python -m luna_genai`를 사용한다.


In [ ]:
import importlib.util
try:
    is_colab = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    is_colab = False
if is_colab:
    app.launch(share=True, debug=False)
else:
    print("로컬 환경: 앱 구성을 확인했습니다. 실제 화면은 python -m luna_genai 로 여세요.")
    app.close()


## 기록하고 마무리

토큰화에서 예상과 달랐던 점 하나, 한 조건 비교 결과 하나, 앱의 callback 연결을 기록한다.
강의노트의 8개 점검 질문을 풀고 별도 해설로 확인한다.
출력은 실행 때 생성되며 배포 노트북에는 저장하지 않는다.
다음 차시에는 같은 GitHub 패키지와 앱에 기능을 추가한다.

출처: 『핸즈온 생성형 AI』 2.1(p.44–67)의 개념 흐름.
코드·리뷰·실습 문제는 수업용 독립 설계.
[Qwen 모델](https://huggingface.co/Qwen/Qwen2-0.5B) ·
[Transformers 생성](https://huggingface.co/docs/transformers/main/en/generation_strategies) ·
[Gradio Blocks](https://github.com/gradio-app/gradio/blob/main/guides/03_building-with-blocks/01_blocks-and-event-listeners.md)


In [ ]:
experiment_record = {
    "package_ref": PUBLIC_REF,
    "model_revision": MODEL_REVISION,
    "mode": lab.device if lab is not None else "numeric-only",
    "tokenization_observation": "",
    "comparison": comparison_note,
    "app_connection": "",
    "unresolved_question": "",
}
experiment_record


## 실행 결과 점검
실제 모델 실험·평가·앱 기능을 확인한다. 수치 실험 전용 모드였다면 전체 실습 검증은
통과하지 않는다. 네트워크를 복구하고 모델 준비 셀부터 다시 실행한다.


In [ ]:
assert lab is not None, "실제 모델이 준비되지 않아 전체 실습을 검증할 수 없습니다."
assert lab.tokenizer.decode(ids_main) == text_main
assert torch.isclose(probabilities_main.sum(), torch.tensor(1., dtype=torch.float64))
assert len(generation_records) == 5 and all(row["생성 부분"].strip() for row in generation_records)
assert len(evaluation_records) == len(load_reviews("evaluation")) == 4
assert all(row[key] in {"positive", "negative"} for row in evaluation_records for key in ("zero_shot", "two_shot"))
# 실제 등록된 앱 callback을 호출한다. 브라우저 화면 검사는 별도로 수행한다.
registered_callbacks = {fn.api_name: fn.fn for fn in app.fns.values()}
assert {"inspect_text", "generate_text", "classify_review", "numeric_distribution"} <= registered_callbacks.keys()
assert registered_callbacks["inspect_text"]("A cat naps.")[0] == "A cat naps."
assert registered_callbacks["classify_review"](development[0]["review"], False)[0] in {"positive", "negative"}
assert not registered_callbacks["numeric_distribution"](1., 0, 1.).empty
print("실제 모델·생성·평가·앱 callback 검사 통과")
